# TexTag: Multi-label tags and search (model 4)

Our models so far pick **one** class out of 47. The end product is a texture **search** tool, which needs two things:

| Search by | Needs | Example |
|---|---|---|
| **Tag** (type a word) | a score for every tag, not just the top one | "striped" → every striped texture, including striped *and* woven ones |
| **Example image** (find similar) | a way to compare two images | this texture → textures that look like it |

This notebook turns our CNNs into both:

1. **Multi-label tags:** same models, new labels (several tags per image) and a new loss.
2. **Index:** run every image through the model once and save its tag scores and its *embedding* (the numbers before the last layer).
   The Godot plugin works the same way.
3. **Measure search** with precision@10, for SmallCNN, ResNet18 and CLIP.

**Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (~35 min).

Sections:
1. Setup
2. Multi-label EDA
3. Loss: sigmoid instead of softmax
4. SmallCNN multi-label
5. ResNet18 multi-label
6. Build the search index
7. Tagging
8. Tag search
9. Find similar
10. Comparison
11. Save the models
12. Findings

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/textag"):
        !git clone -q https://github.com/codecustard/textag.git /content/textag
    !pip install -q open_clip_torch
    %cd /content/textag/notebooks

In [ ]:
import copy
from pathlib import Path
from timeit import default_timer as timer

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import open_clip
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from textag.data import DATA_DIR
from textag.model import SmallCNN, resnet18, unfreeze, n_params
from textag import multilabel

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = os.environ.get("TEXTAG_EPOCHS")   # env var only used for quick test runs

In [ ]:
def plot_curves(hist, title):
    """Train vs val loss and "top tag correct" per epoch; dashed line = best epoch."""
    h = pd.DataFrame({k: v for k, v in hist.items() if k != "best_epoch"})
    h.index = h.index + 1
    best = hist["best_epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric in zip(axes, ["loss", "acc"]):
        ax.plot(h.index, h[f"train_{metric}"], label="train")
        ax.plot(h.index, h[f"val_{metric}"], label="val")
        ax.axvline(best, color="gray", linestyle="--", label=f"best epoch ({best})")
        ax.set_xlabel("epoch")
        ax.set_title({"loss": "Loss (lower is better)", "acc": "Top tag correct (higher is better)"}[metric])
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def show_images(rows, titles, row_labels=None, size=2.6):
    """Grid of thumbnails. rows: list of lists of image paths."""
    fig, axes = plt.subplots(len(rows), len(rows[0]), figsize=(size * len(rows[0]), size * len(rows) + 0.4), squeeze=False)
    for r, paths in enumerate(rows):
        for c, path in enumerate(paths):
            img = Image.open(path).convert("RGB")
            img.thumbnail((200, 200))
            axes[r, c].imshow(img)
            axes[r, c].set_xticks([])
            axes[r, c].set_yticks([])
            axes[r, c].set_title(titles[r][c], fontsize=9)
        if row_labels:
            axes[r, 0].set_ylabel(row_labels[r], fontsize=11)
    plt.tight_layout()
    plt.show()

## 2. Multi-label EDA

The images are the same ones `01_eda` already explored (sizes, colors, normalization), so only the **labels** are new.

DTD has a second label file, `labels_joint_anno.txt`, with **every** tag people gave each image
(the single-label version keeps only one). Same images, same train / val / test split, only `y` changes:

| | `y` for a striped + woven image |
|---|---|
| Single-label (before) | `45` (one class number) |
| Multi-label (now) | `[0, 0, ..., 1, ..., 1, 0]`: 47 yes/no values |

`MultiLabelDTD` in `src/textag/multilabel.py` wraps the dataset from `data.py` and swaps the labels.

In [ ]:
dl = multilabel.loaders(batch_size=32, num_workers=2)
classes = dl["train"].dataset.classes
targets = {s: dl[s].dataset.targets for s in ("train", "val", "test")}

x, y = next(iter(dl["train"]))
print("one batch of images:", tuple(x.shape))
print("one batch of labels:", tuple(y.shape), " (batch, 47 tags)")

### Tags per image, per split

How many images have 1, 2, 3 or 4 tags, and is that the same in train, val and test?

In [ ]:
per_split = {}
for split in ("train", "val", "test"):
    tags_per_image = pd.Series(targets[split].sum(dim=1).int().numpy())
    per_split[split] = tags_per_image.value_counts().sort_index()
per_split = pd.DataFrame(per_split).fillna(0).astype(int).rename_axis("tags per image")
per_split.loc["share with 2+ tags"] = [f"{(targets[s].sum(dim=1) >= 2).float().mean():.1%}" for s in ("train", "val", "test")]
per_split

### Tag frequency

In the single-label data every class has exactly 40 training images. With all tags counted, that changes.

In [ ]:
tag_counts = pd.Series(targets["train"].sum(dim=0).int().numpy(), index=classes).sort_values(ascending=False)

plt.figure(figsize=(13, 4))
plt.bar(tag_counts.index, tag_counts.values, color="#2F5E96")
plt.axhline(40, color="gray", linestyle="--", label="single-label: 40 per class")
plt.xticks(rotation=90)
plt.ylabel("training images with this tag")
plt.title("Tag frequency (train)")
plt.legend()
plt.tight_layout()
plt.show()

print(f"fewest: {tag_counts.min()} images, most: {tag_counts.max()} ({tag_counts.index[0]})")
print("\ntags that gained the most extra images:")
print((tag_counts - 40).head(5).to_string())

### Yes-rate per tag

Each image has ~1.2 tags out of 47, so for any one tag almost every answer is "no".

In [ ]:
yes_rate = targets["train"].mean().item()
print(f"average share of images that have a given tag: {yes_rate:.1%}")
print(f"'accuracy' of a model that always answers no:  {1 - yes_rate:.1%}")

### Which tags appear together

Count every pair of tags that appear on the same image.

In [ ]:
pair_counts = {}
for target in targets["train"]:
    tag_ids = target.nonzero().flatten().tolist()
    for a in range(len(tag_ids)):
        for b in range(a + 1, len(tag_ids)):
            pair = (classes[tag_ids[a]], classes[tag_ids[b]])
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

top_pairs = pd.Series(pair_counts).sort_values(ascending=False).head(10)
top_pairs.rename_axis(["tag 1", "tag 2"]).rename("training images").to_frame()

In [ ]:
# One example image for each of the 6 most common pairs
train_files = dl["train"].dataset.files
paths, titles = [], []
for tag_a, tag_b in top_pairs.index[:6]:
    both = (targets["train"][:, classes.index(tag_a)] == 1) & (targets["train"][:, classes.index(tag_b)] == 1)
    i = both.nonzero().flatten()[0].item()
    paths.append(train_files[i])
    titles.append(f"{tag_a} + {tag_b}")
show_images([paths], [titles])

**Caveat:** the extra tags are **incomplete**. Most images have only 1 tag even when a second one would fit
(many "striped" images are also "banded"). So when a model predicts a tag that isn't listed, it's not always wrong,
and every score below is a slight **underestimate**.

### What the EDA decided

- **Multi-label is worth doing, with a caveat:** about 15% of images have 2+ tags, and the extra tags are incomplete,
  so scores are slight underestimates.
- **No re-splitting needed:** train, val and test have a similar share of multi-tag images (15–17%).
- **Average every metric per tag (macro):** tag counts are no longer balanced (40 to ~100 per tag in train),
  so frequent tags like *blotchy* shouldn't dominate the score.
- **No accuracy-style metric for the tags:** only ~2.5% of answers are "yes", so always answering "no" would score
  97.5%. We use mAP, precision@10 and F1 instead, and pick the tag threshold on validation instead of using 0.5.
- **Multi-label fits the look-alike classes:** the most common pair, dotted + polka-dotted, and pitted + porous
  are the two pairs our single-label model confused most in `04_evaluate`. With multi-label they aren't mistakes;
  an image can be both.

## 3. Loss: sigmoid instead of softmax

| | Single-label (before) | Multi-label (now) |
|---|---|---|
| Question per image | "Which **one** of 47?" | "Yes or no, for **each** of 47?" |
| Last-layer scores become | softmax: 47 probabilities that **add up to 100%** | sigmoid: 47 independent probabilities, each 0–100% |
| Loss | `CrossEntropyLoss` | `BCEWithLogitsLoss` (binary cross-entropy = 47 yes/no losses, averaged) |
| Model | unchanged | unchanged (same last layer, 47 outputs) |

With softmax, a striped + woven image can get at most ~50% each, because the tags compete.
With sigmoid it can be 90% striped **and** 85% woven.

In [ ]:
scores = torch.tensor([[3.0, 2.8, -2.0]])          # last-layer scores for 3 tags: striped, woven, dotted
print("softmax:", torch.softmax(scores, dim=1).numpy().round(2), "  (adds up to 1)")
print("sigmoid:", torch.sigmoid(scores).numpy().round(2), "  (each on its own)")

## 4. SmallCNN multi-label

Same model and settings as `02_baseline`, only the loss changes. Training loop: `multilabel.train`,
the same as `engine.train`, except accuracy is **"top tag correct"** (the highest-scoring tag is one of the image's tags).

| Choice | Value |
|---|---|
| Loss | `BCEWithLogitsLoss` |
| Optimizer | `AdamW`, lr 1e-3, weight decay 1e-4 |
| Epochs | up to 40, early stopping patience 7 |
| LR scheduler | `ReduceLROnPlateau`, halve after 3 stalled epochs |

In [ ]:
torch.manual_seed(42)
small = SmallCNN(num_classes=len(classes))
print(f"trainable parameters: {n_params(small):,}")

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(small.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist_small = multilabel.train(model=small,
                              train_dataloader=dl["train"],
                              val_dataloader=dl["val"],
                              optimizer=optimizer,
                              loss_fn=loss_fn,
                              epochs=int(EPOCHS or 40),
                              device=device,
                              scheduler=scheduler,
                              patience=7)
time_small = timer() - start_time
print(f"SmallCNN training time: {time_small:.1f} seconds")

In [ ]:
plot_curves(hist_small, "SmallCNN multi-label")

## 5. ResNet18 multi-label

Same two phases and settings as `03_resnet`, only the loss changes.

| | Phase 1: feature extractor | Phase 2: fine-tuning |
|---|---|---|
| What trains | Only the new last layer | All layers |
| Learning rate | 1e-3, up to 15 epochs | 1e-4, up to 20 epochs |
| Early stopping | patience 5 | patience 5 |

In [ ]:
torch.manual_seed(42)
resnet = resnet18(num_classes=len(classes), pretrained=True, freeze_backbone=True)

optimizer = torch.optim.AdamW([p for p in resnet.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist_p1 = multilabel.train(model=resnet,
                           train_dataloader=dl["train"],
                           val_dataloader=dl["val"],
                           optimizer=optimizer,
                           loss_fn=loss_fn,
                           epochs=int(EPOCHS or 15),
                           device=device,
                           scheduler=scheduler,
                           patience=5)
time_p1 = timer() - start_time
print(f"Phase 1 training time: {time_p1:.1f} seconds")

In [ ]:
unfreeze(resnet)
optimizer = torch.optim.AdamW(resnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist_p2 = multilabel.train(model=resnet,
                           train_dataloader=dl["train"],
                           val_dataloader=dl["val"],
                           optimizer=optimizer,
                           loss_fn=loss_fn,
                           epochs=int(EPOCHS or 20),
                           device=device,
                           scheduler=scheduler,
                           patience=5)
time_p2 = timer() - start_time
print(f"Phase 2 training time: {time_p2:.1f} seconds")

In [ ]:
plot_curves(hist_p1, "ResNet18 multi-label, phase 1")
plot_curves(hist_p2, "ResNet18 multi-label, phase 2")

## 6. Build the search index

Run every image through each model **once** and keep two things per image:

- **Tag scores:** 47 sigmoid probabilities (for tagging and tag search)
- **Embedding:** the numbers just before the last layer (256 for SmallCNN, 512 for ResNet18), scaled to length 1,
  so `a @ b.T` gives similarity scores (cosine similarity, as in `06_clip`)

After this, searching needs no model at all, only these saved numbers. That's what makes search instant
(and it's exactly what the Godot plugin does with a project's textures).

For comparison, **CLIP** (frozen, no training): its 512 image numbers are the embedding, and the tag scores are the
similarity to the 47 sentences "a {tag} texture." (zero-shot, from `06_clip`).

In [ ]:
small_val = multilabel.index(small, dl["val"], device)        # (tag scores, embeddings)
small_test = multilabel.index(small, dl["test"], device)
resnet_val = multilabel.index(resnet, dl["val"], device)
resnet_test = multilabel.index(resnet, dl["test"], device)
print("SmallCNN test:", tuple(small_test[0].shape), tuple(small_test[1].shape))
print("ResNet18 test:", tuple(resnet_test[0].shape), tuple(resnet_test[1].shape))

In [ ]:
clip_model, _, clip_transform = open_clip.create_model_and_transforms("ViT-B-32-quickgelu", pretrained="openai")
clip_model = clip_model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")

def make_unit_length(x):
    return x / x.norm(dim=1, keepdim=True)

def clip_index(split):
    """CLIP's (tag scores, embeddings) for a split, same image order as our loaders."""
    dataset = datasets.DTD(DATA_DIR, split=split, partition=1, transform=clip_transform)
    all_features = []
    with torch.inference_mode():
        for X, _ in DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2):
            all_features.append(clip_model.encode_image(X.to(device)).float().cpu())
        text_features = clip_model.encode_text(tokenizer([f"a {c} texture." for c in classes]).to(device)).float().cpu()
    image_features = make_unit_length(torch.cat(all_features))
    tag_scores = image_features @ make_unit_length(text_features).T    # (images, 47)
    return tag_scores, image_features

clip_val = clip_index("val")
clip_test = clip_index("test")
test_files = dl["test"].dataset.files
print("CLIP test:", tuple(clip_test[0].shape), tuple(clip_test[1].shape))

## 7. Tagging

**Mean average precision (mAP):** for each tag, rank all test images by that tag's score and check how high the
images that really have it end up (1.0 = all of them at the very top). Averaged over the 47 tags.
It doesn't need a threshold, so it compares the models fairly.

To actually show tags in an app we need a **threshold** (show every tag above X%). It's picked on **validation**
(the one with the best F1), then used once on test.

In [ ]:
def best_threshold(val_scores, val_targets):
    """Threshold with the highest F1 on validation (F1 over all image-tag pairs)."""
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.05, 0.95, 0.05):
        f1 = f1_score(val_targets, val_scores >= t, average="micro", zero_division=0)
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    return best_t

rows = []
for name, val, test in [("SmallCNN", small_val, small_test), ("ResNet18", resnet_val, resnet_test)]:
    t = best_threshold(val[0].numpy(), targets["val"].numpy())
    predicted = test[0].numpy() >= t
    rows.append({"model": name,
                 "test mAP": average_precision_score(targets["test"], test[0], average="macro"),
                 "threshold (from val)": round(t, 2),
                 "precision": precision_score(targets["test"], predicted, average="micro", zero_division=0),
                 "recall": recall_score(targets["test"], predicted, average="micro", zero_division=0),
                 "F1": f1_score(targets["test"], predicted, average="micro", zero_division=0),
                 "tags shown per image": predicted.sum(axis=1).mean()})
rows.append({"model": "CLIP zero-shot", "test mAP": average_precision_score(targets["test"], clip_test[0], average="macro")})
tagging = pd.DataFrame(rows).set_index("model")
thresholds = tagging["threshold (from val)"]
tagging.style.format({"test mAP": "{:.1%}", "precision": "{:.1%}", "recall": "{:.1%}", "F1": "{:.1%}",
                      "threshold (from val)": "{:.2f}", "tags shown per image": "{:.2f}"}, na_rep="-")

**ResNet18 tags on test images that have 2+ true tags** (tags above the threshold, best first):

In [ ]:
t = thresholds["ResNet18"]
multi_test = [i for i in range(len(test_files)) if targets["test"][i].sum() >= 2]
multi_test = multi_test[::len(multi_test) // 6][:6]   # 6 spread-out examples
titles = []
for i in multi_test:
    scores = resnet_test[0][i]
    shown = [(classes[j], scores[j].item()) for j in scores.argsort(descending=True).tolist() if scores[j] >= t][:3]
    true = ", ".join(classes[j] for j in targets["test"][i].nonzero().flatten().tolist())
    titles.append("true: " + true + "\n" + "\n".join(f"{c} {p:.0%}" for c, p in shown))
show_images([[test_files[i] for i in multi_test]], [titles], size=2.9)

## 8. Tag search

Type a tag, get the 10 test images with the highest score for it.
**Precision@10** = how many of those 10 really have the tag, averaged over all 47 tags.
**Random** = picking 10 images at random.

In [ ]:
search_rows = {
    "random": targets["test"].mean().item(),
    "SmallCNN": multilabel.tag_search_precision(small_test[0], targets["test"]).mean().item(),
    "ResNet18": multilabel.tag_search_precision(resnet_test[0], targets["test"]).mean().item(),
    "CLIP zero-shot": multilabel.tag_search_precision(clip_test[0], targets["test"]).mean().item(),
}
pd.Series(search_rows, name="tag search precision@10").map("{:.1%}".format).to_frame()

In [ ]:
queries = ["striped", "honeycombed", "cracked", "marbled"]
rows, titles = [], []
for q in queries:
    top5 = resnet_test[0][:, classes.index(q)].topk(5).indices.tolist()
    rows.append([test_files[i] for i in top5])
    titles.append(["✓" if targets["test"][i, classes.index(q)] else "✗ " + test_files[i].parent.name for i in top5])
print("ResNet18 multi-label: top 5 test images per tag (✓ = has the tag)")
show_images(rows, titles, row_labels=[f'"{q}"' for q in queries])

## 9. Find similar

Give an image, get the 10 most similar test images by embedding (cosine similarity). Every test image is used as a query once.
A result counts as correct if it **shares at least one tag** with the query.

In [ ]:
def random_similar_precision(targets):
    """Chance that a random other image shares a tag."""
    shares = (targets @ targets.T) > 0
    n = len(targets)
    return ((shares.sum() - n) / (n * (n - 1))).item()

similar_rows = {
    "random": random_similar_precision(targets["test"]),
    "SmallCNN (256 numbers)": multilabel.similar_search_precision(small_test[1], targets["test"]),
    "ResNet18 (512 numbers)": multilabel.similar_search_precision(resnet_test[1], targets["test"]),
    "CLIP (512 numbers)": multilabel.similar_search_precision(clip_test[1], targets["test"]),
}
pd.Series(similar_rows, name="find-similar precision@10").map("{:.1%}".format).to_frame()

In [ ]:
query_ids = [0, 400, 900, 1500]
for name, embeddings in [("SmallCNN", small_test[1]), ("ResNet18", resnet_test[1])]:
    rows, titles = [], []
    for q in query_ids:
        similarity = embeddings @ embeddings[q]
        similarity[q] = -float("inf")
        top5 = similarity.topk(5).indices.tolist()
        rows.append([test_files[q]] + [test_files[i] for i in top5])
        titles.append([f"QUERY: {test_files[q].parent.name}"] + [test_files[i].parent.name for i in top5])
    print(f"{name}: query image, then its 5 most similar test images")
    show_images(rows, titles, size=2.3)

## 10. Comparison

In [ ]:
summary = pd.DataFrame({
    "trained on DTD": ["428k weights", "11.2M weights", "nothing (zero-shot)"],
    "tagging mAP": [tagging.loc[m, "test mAP"] for m in ["SmallCNN", "ResNet18", "CLIP zero-shot"]],
    "tag search P@10": [search_rows[m] for m in ["SmallCNN", "ResNet18", "CLIP zero-shot"]],
    "find similar P@10": [similar_rows[m] for m in ["SmallCNN (256 numbers)", "ResNet18 (512 numbers)", "CLIP (512 numbers)"]],
}, index=["SmallCNN", "ResNet18", "CLIP"])
summary.style.format({c: "{:.1%}" for c in ["tagging mAP", "tag search P@10", "find similar P@10"]})

In [ ]:
x = np.arange(3)
fig, ax = plt.subplots(figsize=(9, 4))
for k, (col, color) in enumerate(zip(["tagging mAP", "tag search P@10", "find similar P@10"], ["#2F5E96", "#7FA6D6", "#F0924F"])):
    bars = ax.bar(x + (k - 1) * 0.27, summary[col], width=0.27, label=col, color=color)
    ax.bar_label(bars, labels=[f"{v:.0%}" for v in summary[col]], fontsize=9)
ax.set_xticks(x, summary.index)
ax.set_ylim(0, 1)
ax.set_title("Search quality on the test set")
ax.legend()
plt.show()

## 11. Save the models

New files; the single-label models (`textag.pt`, `baseline.pt`) are not touched.

In [ ]:
MODELS = ROOT / "models"
MODELS.mkdir(exist_ok=True)
multilabel.save_checkpoint(small, "small_cnn", classes, MODELS / "smallcnn_multilabel.pt")
multilabel.save_checkpoint(resnet, "resnet18", classes, MODELS / "textag_multilabel.pt")
for f in ["smallcnn_multilabel.pt", "textag_multilabel.pt"]:
    print(f"saved {f}, {(MODELS / f).stat().st_size / 1e6:.1f} MB")

if IN_COLAB:   # download the weights so they can go into the repo's models/ folder
    from google.colab import files
    files.download(str(MODELS / "smallcnn_multilabel.pt"))
    files.download(str(MODELS / "textag_multilabel.pt"))

## 12. Findings

*(Filled in after the full run.)*